# Trimming della coda 3′
Taglia la coda 3′ di una read alla prima base, letta dalla fine, con qualità almeno pari alla soglia
scelta dall'utente. Si applica allo stesso modo a R1 e a R2, una read alla volta.

Il file contiene due funzioni:
- `phred_scores`: converte la riga di qualità da caratteri ASCII a punteggi numerici;
- `trim_3prime`: usa quei punteggi per decidere dove tagliare e restituisce sequenza e qualità tagliate.

Nel flusso del programma sta fra la lettura e validazione dei file (che consegnano sequenza e
qualità di ogni read) e il controllo di lunghezza (che riceve le read già tagliate).

In [1]:
def phred_scores(quality, offset=33):
    """I punteggi di qualità Phred di una riga di qualità FASTQ, uno per base.

    Ogni carattere della riga codifica un numero: il suo codice ASCII meno 33
    (codifica Phred+33, quella di Illumina). "!" vale 0, "5" vale 20, "I" vale 40.
    Il punteggio Q corrisponde a una probabilità di errore di 10^(-Q/10):
    Q = 20 significa un errore ogni 100 basi.

    **Perché ord.** In Python un carattere è una stringa lunga 1, non un numero:
    ord() ne restituisce il codice ASCII, su cui si può fare la sottrazione.
    """
    scores = []
    for char in quality:
        scores.append(ord(char) - offset)
    return scores


In [2]:
def trim_3prime(sequence, quality, qmin=20):
    """La read con la coda 3' di bassa qualità tagliata: (sequenza, qualità).

    La read si legge dalla fine verso l'inizio (slicing [::-1]) e si contano
    le basi con qualità sotto qmin, fermandosi alla prima base che raggiunge
    la soglia: quella e tutte le precedenti si tengono. Il taglio è uguale per
    la sequenza e per la riga di qualità, così le due restano allineate.

    **Solo la coda.** Una base di bassa qualità in mezzo alla read non viene
    toccata: il trimming 3' rimuove soltanto le basi scadenti all'estremità,
    dove la qualità del sequenziamento Illumina tende a calare.

    **La soglia.** Una base si tiene se la sua qualità è maggiore o uguale a
    qmin. Il valore predefinito è 20, quello delle slide (Qmin = 20); chi chiama
    la funzione passa la soglia scelta dall'utente.

    **Casi limite.** Se nessuna base della coda è sotto soglia la read resta
    intera; se lo sono tutte, sequenza e qualità diventano stringhe vuote, e
    sarà il controllo di lunghezza a scartare la read.

    **Chi la usa.** Il programma principale, una volta per R1 e una per R2 di
    ogni coppia, prima del controllo di lunghezza:
    seq_r1, qual_r1 = trim_3prime(seq_r1, qual_r1, qmin)
    """
    # Si tolgono eventuali "\n" o spazi ai lati, nel caso le righe arrivino
    # dal file senza essere state pulite: un "\n" verrebbe letto come una base.
    sequence = sequence.strip()
    quality = quality.strip()

    scores = phred_scores(quality)

    # Quante basi togliere dalla fine: si scorrono i punteggi al contrario
    # e ci si ferma alla prima base che raggiunge la soglia.
    n_cut = 0
    for score in scores[::-1]:
        if score >= qmin:
            break
        n_cut += 1

    # Si taglia fino a "lunghezza - basi da togliere". Si evita sequence[:-n_cut]
    # perché con n_cut = 0 diventerebbe sequence[:-0] = sequence[:0], cioè
    # una stringa vuota: una read perfetta verrebbe cancellata invece che tenuta.
    end = len(sequence) - n_cut
    return sequence[:end], quality[:end]


## Verifiche
Le celle non richiedono input: basta eseguirle in ordine, dopo le due definizioni.

### La riga di qualità diventa una lista di numeri
**Obiettivo**: Verificare la conversione ASCII → Phred sui caratteri che corrispondono a 0, 10, 20, 30 e 40.

**Atteso**: `[0, 10, 20, 30, 40]`

In [3]:
print(phred_scores("!+5?I"))

[0, 10, 20, 30, 40]


### Una coda di bassa qualità viene tagliata
**Obiettivo**: Verificare che le ultime tre basi, con qualità 2, 3 e 4, vengano rimosse e che la riga di qualità venga tagliata nello stesso punto.

**Atteso**: `('ACGTA', 'IIIII')`

In [4]:
print(trim_3prime("ACGTACGT", "IIIII#$%", 20))

('ACGTA', 'IIIII')


### Una read senza coda scadente resta intera
**Obiettivo**: Verificare il caso in cui non c'è niente da tagliare: è il caso che `sequence[:-0]` sbaglierebbe.

**Atteso**: `('ACGT', 'IIII')`

In [5]:
print(trim_3prime("ACGT", "IIII", 20))

('ACGT', 'IIII')


### Una read tutta sotto soglia diventa vuota
**Obiettivo**: Verificare che, se nessuna base raggiunge la soglia, la funzione restituisca due stringhe vuote senza andare in errore.

**Atteso**: `('', '')`

In [6]:
print(trim_3prime("ACGT", "!!#$", 20))

('', '')


### Una base scadente in mezzo alla read non si taglia
**Obiettivo**: Verificare che la funzione tagli solo la coda: la base centrale con qualità 2 resta, perché dopo di lei ci sono basi buone.

**Atteso**: `('ACGTA', 'II#II')`

In [7]:
print(trim_3prime("ACGTA", "II#II", 20))

('ACGTA', 'II#II')


### Una base con qualità esattamente pari alla soglia si tiene
**Obiettivo**: Verificare che il confronto sia "maggiore o uguale": il carattere `5` vale esattamente 20.

**Atteso**: `('ACG', 'II5')`

In [8]:
print(trim_3prime("ACGT", "II5#", 20))

('ACG', 'II5')


### La soglia scelta dall'utente cambia il taglio
**Obiettivo**: Verificare che la stessa read venga tagliata diversamente con due soglie: l'ultima base vale 30, quindi si tiene con soglia 20 e si toglie con soglia 35.

**Atteso**: `('ACGTA', 'IIII?')` con soglia 20, poi `('ACGT', 'IIII')` con soglia 35

In [9]:
print(trim_3prime("ACGTA", "IIII?", 20))
print(trim_3prime("ACGTA", "IIII?", 35))

('ACGTA', 'IIII?')
('ACGT', 'IIII')


### Le righe con l'a capo vengono pulite
**Obiettivo**: Verificare che un `\n` lasciato in fondo alle righe lette dal file non venga scambiato per una base.

**Atteso**: `('ACGTA', 'IIIII')`

In [10]:
print(trim_3prime("ACGTACGT\n", "IIIII#$%\n", 20))

('ACGTA', 'IIIII')


### Una coppia R1 / R2
**Obiettivo**: Verificare l'uso previsto nel programma principale: la stessa funzione e la stessa soglia applicate ai due mate, con lunghezze iniziali e finali utili al controllo di lunghezza e al report.

**Atteso**: R1 passa da 10 a 7 basi, R2 da 10 a 10

In [11]:
qmin = 20
seq_r1, qual_r1 = "ACGTACGTAC", "IIIIIII##!"
seq_r2, qual_r2 = "TTGACCATGG", "IIIIIIIIII"

trimmed_r1, trimmed_qual_r1 = trim_3prime(seq_r1, qual_r1, qmin)
trimmed_r2, trimmed_qual_r2 = trim_3prime(seq_r2, qual_r2, qmin)

print("R1:", len(seq_r1), "->", len(trimmed_r1), trimmed_r1)
print("R2:", len(seq_r2), "->", len(trimmed_r2), trimmed_r2)

R1: 10 -> 7 ACGTACG
R2: 10 -> 10 TTGACCATGG
